# 03 — Requêtage : SELECT, INSERT, UPDATE, DELETE, transactions

Ce notebook couvre **toutes les techniques de requêtage** du repo, avec
systématiquement deux approches quand c'est pertinent :
- **(A) SQL brut** via SQLAlchemy (flexible, fonctionne pour toutes les bases)
- **(B) SDK Supabase** (API REST auto-générée, pratique pour du CRUD simple)

⚠️ Toutes les requêtes montrées sont **paramétrées** — jamais de f-string
avec une valeur utilisateur dans le SQL (protection anti-injection).


In [ ]:
import sys
sys.path.append("..")

from db_connection.sqlite_connector import get_sqlite_engine
engine = get_sqlite_engine()

# Table de démo pour ce notebook
from sqlalchemy import text
with engine.begin() as conn:
    conn.execute(text('''
        CREATE TABLE IF NOT EXISTS demo_products (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            category TEXT,
            price REAL,
            stock INTEGER
        )
    '''))
print("Base de démo prête.")


## 1. SELECT — requêtes de lecture

In [ ]:
from queries.select_queries import select_all, select_with_filter, select_paginated

# D'abord on insère quelques données de test
from queries.write_queries import bulk_insert

bulk_insert(engine, "demo_products", [
    {"name": "Clavier mécanique", "category": "Informatique", "price": 89.99, "stock": 12},
    {"name": "Souris sans fil", "category": "Informatique", "price": 29.99, "stock": 40},
    {"name": "Chaise de bureau", "category": "Mobilier", "price": 149.00, "stock": 5},
])


In [ ]:
# SELECT * simple
rows = select_all(engine, "demo_products")
rows


In [ ]:
# SELECT filtré (requête paramétrée)
rows_filtre = select_with_filter(engine, "demo_products", column="category", value="Informatique")
rows_filtre


In [ ]:
# Pagination (utile pour de gros volumes)
page_1 = select_paginated(engine, "demo_products", page=1, page_size=2)
page_1


## 2. INSERT / UPDATE / DELETE — requêtes d'écriture

In [ ]:
from queries.write_queries import insert_row, update_rows, delete_rows

# INSERT
insert_row(engine, "demo_products", {"name": "Écran 27 pouces", "category": "Informatique", "price": 249.0, "stock": 8})


In [ ]:
# UPDATE (avec filtre)
nb_modifiees = update_rows(engine, "demo_products", updates={"stock": 3}, where_column="name", where_value="Chaise de bureau")
print(f"{nb_modifiees} ligne(s) modifiée(s).")


In [ ]:
# DELETE
nb_supprimees = delete_rows(engine, "demo_products", where_column="name", where_value="Souris sans fil")
print(f"{nb_supprimees} ligne(s) supprimée(s).")


## 3. Transactions — plusieurs opérations "tout ou rien"

Exemple : si une des étapes échoue, TOUT est annulé automatiquement (rollback).


In [ ]:
from queries.transactions import run_in_transaction

operations = [
    ("UPDATE demo_products SET stock = stock - :qty WHERE name = :name", {"qty": 1, "name": "Clavier mécanique"}),
    ("INSERT INTO demo_products (name, category, price, stock) VALUES (:name, :cat, :price, :stock)",
     {"name": "Tapis de souris", "cat": "Informatique", "price": 9.99, "stock": 100}),
]

run_in_transaction(engine, operations)
print("Transaction exécutée avec succès.")


## 4. Équivalent via le SDK Supabase (API REST)

Si tu utilises Supabase, voici l'équivalent de ces opérations via le client SDK
(remplace `demo_products` par ta vraie table, et utilise `get_supabase_client()`
au lieu d'un moteur SQL) :

```python
from db_connection.supabase_connector import get_supabase_client
from queries.select_queries import supabase_select_all, supabase_select_with_filter
from queries.write_queries import supabase_insert, supabase_update, supabase_delete

client = get_supabase_client()

# SELECT
rows = supabase_select_all(client, "products")

# INSERT
supabase_insert(client, "products", {"name": "Nouveau produit", "price": 19.99})

# UPDATE
supabase_update(client, "products", {"stock": 0}, column="name", value="Souris sans fil")

# DELETE
supabase_delete(client, "products", column="id", value=42)
```


## Suite

➡️ Passe au notebook **`04_exploitation_resultats.ipynb`** : c'est LE notebook central
qui montre comment transformer, analyser, visualiser et exporter les résultats
obtenus dans ce notebook.
